# S1 — Comprendre avant de coder
## Python pour la Data Science — ESEO Pro

> **Fil rouge :** un code qui s’exécute sans erreur n’est pas une preuve que le résultat est vrai.

Cette séance suit le support **Logic Over Code**. Le notebook sert aux manipulations : environnement, NumPy, Data Detective, grain et piège de jointure.

### Objectifs
- vérifier un environnement Python reproductible ;
- manipuler des tableaux NumPy sans tunnel de syntaxe ;
- transformer une question métier en mesure vérifiable ;
- identifier le **grain** d’une table ;
- comprendre pourquoi une jointure techniquement valide peut produire un résultat métier faux ;
- utiliser l’IA comme copilote, jamais comme preuve.


## 0 — Avant Python : la question

**Challenge oral** : « Quel produit se vend le mieux ? »

Avant de coder, notez votre définition de **se vend le mieux** : quantité ? chiffre d’affaires ? marge ? nombre de commandes ?

**Règle :** mesurable ≠ pertinent.


## 1 — Build Your Lab : vérification

Le dépôt fournit `setup.sh`, `check.sh` et `run.sh`. Cette cellule vérifie seulement le kernel réellement utilisé par Jupyter.


In [3]:
import sys, platform
from pathlib import Path
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())

ROOT = Path.cwd()
while not (ROOT / "datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "datasets").exists(), "Ouvrez ce notebook depuis le dépôt ai-course."
print("Repo root:", ROOT)


Python: 3.14.7
Executable: C:\Users\Mouss\OneDrive\Desktop\ai-course-student\.venv\Scripts\python.exe
Platform: Windows-11-10.0.26200-SP0
Repo root: C:\Users\Mouss\OneDrive\Desktop\ai-course-student


In [4]:
import numpy as np
import pandas as pd
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("✅ Environnement prêt")


NumPy: 2.5.3
Pandas: 3.0.5
✅ Environnement prêt


## 2 — IA locale : démonstration professeur

Prompt de démonstration Ollama :

> « Écris un programme Python qui charge `sales.csv` et calcule le chiffre d’affaires par produit. »

**Ne copiez pas aveuglément la réponse.** Demandez :
- que signifie « chiffre d’affaires » dans les colonnes disponibles ?
- y a-t-il une quantité ? des retours ? des doublons ?
- comment vérifier le total indépendamment ?

Workflow : **Generate → Run → Inspect → Question → Validate**.


## 3 — NumPy : le moteur numérique

On part d’un problème simple : prix × quantités. L’objectif n’est pas de mémoriser l’API, mais de comprendre la vectorisation.


In [5]:
prices = np.array([12.0, 25.0, 8.0, 40.0])
quantities = np.array([3, 1, 5, 2])
revenue_by_line = prices * quantities
revenue_by_line


array([36., 25., 40., 80.])

In [6]:
print("shape:", revenue_by_line.shape)
print("dtype:", revenue_by_line.dtype)
print("total:", revenue_by_line.sum())
print("mean:", revenue_by_line.mean())
print("max:", revenue_by_line.max())
assert np.isclose(revenue_by_line.sum(), 181.0)
print("✅ Contrôle indépendant attendu : 181.0")


shape: (4,)
dtype: float64
total: 181.0
mean: 45.25
max: 80.0
✅ Contrôle indépendant attendu : 181.0


### Mini-challenge NumPy
Sans boucle `for`, calculez le prix TTC avec +20 %, puis le CA TTC par ligne. Expliquez ce que fait chaque opération.


In [7]:
# Correction à dévoiler après tentative
prices_ttc = prices * 1.20
revenue_ttc = prices_ttc * quantities
print(prices_ttc)
print(revenue_ttc)
print("CA TTC:", revenue_ttc.sum())


[14.4 30.   9.6 48. ]
[43.2 30.  48.  96. ]
CA TTC: 217.2


## 4 — Data Detective

Mission par équipe : **« Quel produit se vend le mieux ? »**

Avant tout calcul :
1. définissez « se vend le mieux » ;   celui qui se vend le plus c'est celui se vend le plus . le produit que les clients achetent le plus
2. dites ce que représente **une ligne** de chaque fichier ; customers.csv -> un client . orders.csv -> une commande . products.csv -> un produit du cataloque . order_lines.csv -> une ligne de commande ;
3. identifiez les clés ; ( customer_id , order_id , product_id , lines_id )
4. choisissez une mesure ; la quantité totale d’unités vendues par produit.
5. prévoyez un contrôle indépendant.   vérifier une deuxième fois que le produit ayant le plus grand total de quantités vendues est bien le produit de base .


### Stop avant de continuer
Écrivez vos réponses.

Grains attendus :
- `customers` : une ligne = un client ;
- `orders` : une ligne = une commande ;
- `order_lines` : une ligne = un produit dans une commande ;
- `products` : une ligne = un produit.

La table de faits naturelle pour le CA est au **grain ligne de commande**.


In [8]:
DATA = ROOT / "datasets" / "retail_case"
customers = pd.read_csv(DATA / "customers.csv")
orders = pd.read_csv(DATA / "orders.csv")
order_lines = pd.read_csv(DATA / "order_lines.csv")
products = pd.read_csv(DATA / "products.csv")

for name, df in {"customers": customers, "orders": orders, "order_lines": order_lines, "products": products}.items():
    print(f"{name:12s} shape={df.shape} columns={list(df.columns)}")
    display(df.head(3))


customers    shape=(5, 4) columns=['customer_id', 'customer_name', 'city', 'segment']


,customer_id,customer_name,city,segment
0,C001,Alice,Niort,Consumer
1,C002,Benoit,Angers,SMB
2,C003,Chloe,Nantes,Consumer


orders       shape=(6, 4) columns=['order_id', 'customer_id', 'order_date', 'channel']


,order_id,customer_id,order_date,channel
0,O1001,C001,2026-01-05,web
1,O1002,C002,2026-01-05,store
2,O1003,C001,2026-01-06,web


order_lines  shape=(8, 5) columns=['order_id', 'line_id', 'product_id', 'quantity', 'unit_price']


,order_id,line_id,product_id,quantity,unit_price
0,O1001,1,P01,1,45
1,O1001,2,P04,3,8
2,O1002,1,P02,1,90


products     shape=(4, 4) columns=['product_id', 'product_name', 'category', 'unit_price']


,product_id,product_name,category,unit_price
0,P01,Laptop Stand,Office,45
1,P02,Mechanical Keyboard,Electronics,90
2,P03,USB-C Hub,Electronics,55


In [9]:
print("Clients :", customers["customer_id"].nunique(), "identifiants uniques sur", len(customers), "lignes")
print("Commandes :", orders["order_id"].nunique(), "identifiants uniques sur", len(orders), "lignes")
print("Produits :", products["product_id"].nunique(), "identifiants uniques sur", len(products), "lignes")

duplicates_lines = order_lines.duplicated(
    subset=["order_id", "line_id"]
).sum()

print("Doublons dans les lignes de commande :", duplicates_lines)

Clients : 5 identifiants uniques sur 5 lignes
Commandes : 6 identifiants uniques sur 6 lignes
Produits : 4 identifiants uniques sur 4 lignes
Doublons dans les lignes de commande : 0


In [10]:
# Vérifions les hypothèses de clés avant la jointure.
print("customers.customer_id unique:", customers["customer_id"].is_unique)
print("orders.order_id unique:", orders["order_id"].is_unique)
print("products.product_id unique:", products["product_id"].is_unique)
print("order_lines (order_id, line_id) unique:", not order_lines.duplicated(["order_id", "line_id"]).any())


customers.customer_id unique: True
orders.order_id unique: True
products.product_id unique: True
order_lines (order_id, line_id) unique: True


## 5 — Construire le fait de vente

On déclare les cardinalités attendues avec `validate=` : ce n’est pas du confort, c’est une hypothèse métier rendue testable.


In [15]:
fact_sales = (
    order_lines
    .merge(orders, on="order_id", how="left", validate="many_to_one")
    .merge(customers, on="customer_id", how="left", validate="many_to_one")
    .merge(products[["product_id", "product_name", "category"]], on="product_id", how="left", validate="many_to_one")
)
fact_sales["revenue"] = fact_sales["quantity"] * fact_sales["unit_price"]

reference_revenue = fact_sales["revenue"].sum()
print("Rows:", len(fact_sales))
print("Reference revenue:", reference_revenue)
display(fact_sales.head())


Rows: 8
Reference revenue: 634


,order_id,line_id,product_id,quantity,unit_price,customer_id,order_date,channel,customer_name,city,segment,product_name,category,revenue
0,O1001,1,P01,1,45,C001,2026-01-05,web,Alice,Niort,Consumer,Laptop Stand,Office,45
1,O1001,2,P04,3,8,C001,2026-01-05,web,Alice,Niort,Consumer,Notebook,Office,24
2,O1002,1,P02,1,90,C002,2026-01-05,store,Benoit,Angers,SMB,Mechanical Keyboard,Electronics,90
3,O1003,1,P03,2,55,C001,2026-01-06,web,Alice,Niort,Consumer,USB-C Hub,Electronics,110
4,O1004,1,P04,5,8,C003,2026-01-07,web,Chloe,Nantes,Consumer,Notebook,Office,40


### Répondre à la question métier
Voici deux définitions légitimes et potentiellement différentes : quantité vendue et chiffre d’affaires.


In [16]:
by_product = (fact_sales.groupby(["product_id", "product_name"], as_index=False)
              .agg(quantity_sold=("quantity", "sum"), revenue=("revenue", "sum"))
              .sort_values("revenue", ascending=False))
display(by_product)
print("Leader CA:", by_product.iloc[0]["product_name"])
print("Leader quantité:", by_product.sort_values("quantity_sold", ascending=False).iloc[0]["product_name"])


,product_id,product_name,quantity_sold,revenue
1,P02,Mechanical Keyboard,3,270
2,P03,USB-C Hub,3,165
0,P01,Laptop Stand,3,135
3,P04,Notebook,8,64


Leader CA: Mechanical Keyboard
Leader quantité: Notebook


## 6 — Le piège de jointure

Un fichier marketing associe plusieurs tags à certains clients. La jointure suivante **s’exécute parfaitement**. Est-elle correcte pour calculer le CA total ?


In [17]:
customer_tags = pd.DataFrame({
    "customer_id": ["C001", "C001", "C002", "C002", "C003", "C003", "C004", "C004", "C005", "C005"],
    "tag": ["newsletter", "vip", "newsletter", "promo", "organic", "loyalty", "partner", "vip", "organic", "promo"]
})

bad = fact_sales.merge(customer_tags, on="customer_id", how="left")
bad_revenue = bad["revenue"].sum()
print("Rows before:", len(fact_sales), "| after:", len(bad))
print("CA before:", reference_revenue, "| after:", bad_revenue)
print("Inflation:", round(bad_revenue / reference_revenue, 3), "x")
assert len(bad) > len(fact_sales)
assert bad_revenue > reference_revenue
print("⚠️ Aucun crash Python. Résultat métier faux.")


Rows before: 8 | after: 16
CA before: 634 | after: 1268
Inflation: 2.0 x
⚠️ Aucun crash Python. Résultat métier faux.


### Faire échouer explicitement une mauvaise hypothèse
Si vous pensiez que chaque client n’avait qu’un tag, dites-le à Pandas.


In [18]:
try:
    fact_sales.merge(customer_tags, on="customer_id", how="left", validate="many_to_one")
except Exception as e:
    print(type(e).__name__ + ":", e)


MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 customer_id
       C001
       C002
       C003
       C004
       C005 ...


## 7 — Contrat de données minimal

Quelques invariants simples peuvent empêcher une erreur silencieuse.


In [19]:
checks = {
    "customer_id_unique": customers["customer_id"].is_unique,
    "order_id_unique": orders["order_id"].is_unique,
    "product_id_unique": products["product_id"].is_unique,
    "order_line_composite_key_unique": not order_lines.duplicated(["order_id", "line_id"]).any(),
    "quantities_positive": (order_lines["quantity"] > 0).all(),
    "prices_non_negative": (order_lines["unit_price"] >= 0).all(),
    "fact_rows_preserved": len(fact_sales) == len(order_lines),
}
for k, v in checks.items():
    print(("✅" if v else "❌"), k)
assert all(checks.values())


✅ customer_id_unique
✅ order_id_unique
✅ product_id_unique
✅ order_line_composite_key_unique
✅ quantities_positive
✅ prices_non_negative
✅ fact_rows_preserved


## 8 — Exit ticket

Répondez en trois phrases :
1. Quelle question faut-il poser **avant** de coder ?
2. Donnez un exemple de proxy mesurable mais trompeur.
3. Quel résultat obtenu aujourd’hui pourriez-vous vérifier d’une autre manière ?

---

# À retenir

**Mesurable ≠ pertinent.**  
**L’IA peut produire le code. Vous restez responsables du résultat.**  
**Si vous ne savez pas ce qu’une ligne représente, ne calculez rien.**  
**LE CODE TOURNE N’EST PAS UN TEST DE VÉRITÉ.**
